In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!mkdir -p "/content/drive/MyDrive/Iraq_Lake/images"
!mkdir -p "/content/drive/MyDrive/Iraq_Lake/masks"
!mv "/content/drive/MyDrive/Iraq_Lake/"*"_s2_4band"*.tif* "/content/drive/MyDrive/Iraq_Lake/images/" 2>/dev/null || true
!mv "/content/drive/MyDrive/Iraq_Lake/"*"_gsw_water"*.tif* "/content/drive/MyDrive/Iraq_Lake/masks/" 2>/dev/null || true

!echo "IMAGES:"; ls "/content/drive/MyDrive/Iraq_Lake/images" | head -n 20
!echo "MASKS:";  ls "/content/drive/MyDrive/Iraq_Lake/masks"  | head -n 20

image_dir = "/content/drive/MyDrive/Iraq_Lake/images"
mask_dir  = "/content/drive/MyDrive/Iraq_Lake/masks"
!mkdir -p "/content/drive/MyDrive/Iraq_Lake/Iraq_Lakes_s2_water_seg"


IMAGES:
darbandikhan_2021_s2_4band.tif
dukan_2021_s2_4band.tif
habbaniyah_2021_s2_4band.tif
razzaza_2021_s2_4band.tif
tharthar_2021_s2_4band.tif
MASKS:
darbandikhan_2021_gsw_water.tif
dukan_2021_gsw_water.tif
habbaniyah_2021_gsw_water.tif
razzaza_2021_gsw_water.tif
tharthar_2021_gsw_water.tif


In [3]:
import os, glob, random
import numpy as np
import rasterio
import imageio.v2 as imageio
from collections import defaultdict
from rasterio.warp import reproject, Resampling

# ---------------------------
# 0) PATHS + SETTINGS
# ---------------------------
GEE_IMG_DIR = "/content/drive/MyDrive/Iraq_Lake/images"
GEE_MSK_DIR = "/content/drive/MyDrive/Iraq_Lake/masks"
OUT_ROOT    = "/content/drive/MyDrive/Iraq_Lake/Iraq_Lake_s2_water_seg"

TILE = 512
SEED = 42

# If you want 2021 only, keep this. Otherwise set to None to include all years found.
ONLY_YEAR = 2021  # or None

# Target counts
TARGET_TRAIN = 250
TARGET_VAL   = 100
TARGET_TEST  = 20

# Water coverage buckets
WATER_HEAVY_FRAC = 0.15  # >= this => "water-heavy"
P_EMPTY = 0.33
P_MIXED = 0.34
P_HEAVY = 0.33

# ✅ Lake-wise splits
TRAIN_LAKES = {"razzaza", "habbaniyah", "tharthar"}
VAL_LAKES   = {"dukan"}
TEST_LAKES  = {"darbandikhan"}

random.seed(SEED)
np.random.seed(SEED)

# ---------------------------
# 1) HELPERS
# ---------------------------
def make_dirs(root: str) -> None:
    for split in ["training", "validation", "test"]:
        os.makedirs(os.path.join(root, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(root, split, "masks"), exist_ok=True)
        # NEW: RGB preview PNGs (true-colour)
        os.makedirs(os.path.join(root, split, "rgb_png"), exist_ok=True)

def align_mask_to_image(img_path: str, mask_path: str):
    """
    Returns:
      img_arr: (H, W, 4) float32
      mask_arr: (H, W) uint8 in {0,1}
    """
    with rasterio.open(img_path) as src_img:
        img = src_img.read().astype(np.float32)   # (bands,H,W)
        img_transform = src_img.transform
        img_crs = src_img.crs
        img_h, img_w = src_img.height, src_img.width

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    with rasterio.open(mask_path) as src_msk:
        msk = src_msk.read(1).astype(np.float32)
        msk = np.nan_to_num(msk, nan=0.0, posinf=0.0, neginf=0.0)

        same_grid = (
            src_msk.crs == img_crs and
            src_msk.transform == img_transform and
            src_msk.width == img_w and
            src_msk.height == img_h
        )

        if same_grid:
            mask_resampled = msk
        else:
            mask_resampled = np.zeros((img_h, img_w), dtype=np.float32)
            reproject(
                source=msk,
                destination=mask_resampled,
                src_transform=src_msk.transform,
                src_crs=src_msk.crs,
                dst_transform=img_transform,
                dst_crs=img_crs,
                resampling=Resampling.nearest
            )

    img_arr = np.transpose(img, (1, 2, 0))             # (H,W,4)
    mask_arr = (mask_resampled > 0.5).astype(np.uint8) # (H,W)
    return img_arr, mask_arr

def per_tile_minmax(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mn = float(np.min(x))
    mx = float(np.max(x))
    if mx <= mn:
        return np.zeros_like(x, dtype=np.float32)
    return (x - mn) / (mx - mn)

def water_fraction(msk_t: np.ndarray) -> float:
    return float(msk_t.mean())

def parse_lake_year_and_base(s2_path: str):
    """
    Expected:
      razzaza_2021_s2_4band.tif  -> lake='razzaza', year=2021, base='razzaza_2021'
    """
    fname = os.path.basename(s2_path)
    stem = os.path.splitext(fname)[0]
    base = stem.replace("_s2_4band", "")  # e.g., 'razzaza_2021'
    parts = base.split("_")
    year = int(parts[-1])
    lake = "_".join(parts[:-1])
    return lake, year, base

def bucket_tiles(tiles):
    empty = [t for t in tiles if t["wf"] == 0.0]
    heavy = [t for t in tiles if t["wf"] >= WATER_HEAVY_FRAC]
    mixed = [t for t in tiles if (0.0 < t["wf"] < WATER_HEAVY_FRAC)]
    return empty, mixed, heavy

def sample_k(lst, k):
    if k <= 0:
        return []
    k = min(k, len(lst))
    random.shuffle(lst)
    return lst[:k]

def build_split_balanced(tiles_pool, n_total):
    """
    Balanced sampling from tiles_pool:
    - tries P_EMPTY / P_MIXED / P_HEAVY
    - tops up from remaining if a bucket is short
    """
    if n_total <= 0:
        return []

    empty, mixed, heavy = bucket_tiles(tiles_pool)

    n_e = int(round(P_EMPTY * n_total))
    n_m = int(round(P_MIXED * n_total))
    n_h = n_total - n_e - n_m

    chosen = []
    chosen += sample_k(empty.copy(), n_e)
    chosen += sample_k(mixed.copy(), n_m)
    chosen += sample_k(heavy.copy(), n_h)

    if len(chosen) < n_total:
        chosen_ids = set(t["id"] for t in chosen)
        remaining = [t for t in tiles_pool if t["id"] not in chosen_ids]
        chosen += sample_k(remaining, n_total - len(chosen))

    random.shuffle(chosen)
    return chosen

def summarise_split(name, tiles_list):
    wfs = np.array([t["wf"] for t in tiles_list], dtype=np.float32) if tiles_list else np.array([])
    if len(wfs) == 0:
        print(f"{name}: n=0")
        return
    pct_nonempty = float((wfs > 0).mean() * 100.0)
    pct_heavy = float((wfs >= WATER_HEAVY_FRAC).mean() * 100.0)
    lakes = sorted(set(t["lake"] for t in tiles_list))
    years = sorted(set(t["year"] for t in tiles_list))
    print(
        f"{name}: n={len(tiles_list)} | lakes={lakes} | years={years} | "
        f"non-empty={pct_nonempty:.1f}% | heavy(>={WATER_HEAVY_FRAC:.2f})={pct_heavy:.1f}% | wf_mean={wfs.mean():.3f}"
    )

# ---------------------------
# 2) RGB PREVIEW (TRUE COLOUR ONLY)
# ---------------------------
def rgb_preview_uint8(img4: np.ndarray, lo=2, hi=98) -> np.ndarray:
    """
    Assumes 4-band order: [B2,B3,B4,B8] = [Blue,Green,Red,NIR]
    True-colour RGB = [B4,B3,B2] -> indices [2,1,0]
    Per-channel percentile stretch for nicer display.
    """
    rgb = img4[..., [2, 1, 0]].astype(np.float32)

    out = np.zeros_like(rgb, dtype=np.float32)
    for c in range(3):
        p_lo, p_hi = np.percentile(rgb[..., c], (lo, hi))
        out[..., c] = np.clip((rgb[..., c] - p_lo) / (p_hi - p_lo + 1e-6), 0, 1)

    return (out * 255).astype(np.uint8)

# ---------------------------
# 3) SAVE (NPY + RGB PNG PREVIEW)
# ---------------------------
def save_tiles_with_rgb(tiles_list, split_name):
    img_out = os.path.join(OUT_ROOT, split_name, "images")
    msk_out = os.path.join(OUT_ROOT, split_name, "masks")
    rgb_out = os.path.join(OUT_ROOT, split_name, "rgb_png")

    for t in tiles_list:
        x = per_tile_minmax(t["img"]).astype(np.float32)  # (512,512,4)
        y = t["msk"].astype(np.uint8)[..., None]          # (512,512,1)

        np.save(os.path.join(img_out, t["id"] + ".npy"), x)
        np.save(os.path.join(msk_out, t["id"] + ".npy"), y)

        rgb_png = rgb_preview_uint8(t["img"])
        imageio.imwrite(os.path.join(rgb_out, t["id"] + ".png"), rgb_png)

# ---------------------------
# 4) COLLECT TILES (GROUPED BY LAKE)
# ---------------------------
def collect_tiles_grouped_by_lake():
    s2_files = sorted(glob.glob(os.path.join(GEE_IMG_DIR, "*_s2_4band.tif*")))
    if not s2_files:
        raise FileNotFoundError(f"No S2 files found in: {GEE_IMG_DIR}")

    tiles_by_lake = defaultdict(list)

    for s2_path in s2_files:
        lake, year, base = parse_lake_year_and_base(s2_path)

        if ONLY_YEAR is not None and year != ONLY_YEAR:
            continue

        msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tif")
        if not os.path.exists(msk_path):
            msk_path = os.path.join(GEE_MSK_DIR, base + "_gsw_water.tiff")

        if not os.path.exists(msk_path):
            print(f"⚠ Missing mask for {base}, skipping")
            continue

        print("Reading + aligning:", base)
        img_arr, mask_arr = align_mask_to_image(s2_path, msk_path)
        H, W, C = img_arr.shape

        for r0 in range(0, H - TILE + 1, TILE):
            for c0 in range(0, W - TILE + 1, TILE):
                img_t = img_arr[r0:r0+TILE, c0:c0+TILE, :]
                msk_t = mask_arr[r0:r0+TILE, c0:c0+TILE]
                wf = water_fraction(msk_t)

                tile_id = f"{base}_r{r0//TILE:03d}_c{c0//TILE:03d}"

                tiles_by_lake[lake].append({
                    "id": tile_id,
                    "img": img_t,
                    "msk": msk_t,
                    "lake": lake,
                    "year": year,
                    "wf": wf
                })

    if not tiles_by_lake:
        raise RuntimeError("No tiles created. Check your rasters and masks.")
    return tiles_by_lake

# ---------------------------
# 5) BUILD SPLITS BY LAKE (NO LEAKAGE)
# ---------------------------
def build_splits_from_lakes(tiles_by_lake):
    overlap = (TRAIN_LAKES & VAL_LAKES) | (TRAIN_LAKES & TEST_LAKES) | (VAL_LAKES & TEST_LAKES)
    if overlap:
        raise ValueError(f"Lake leakage: these lakes appear in multiple splits: {sorted(overlap)}")

    all_known = set(tiles_by_lake.keys())
    assigned = TRAIN_LAKES | VAL_LAKES | TEST_LAKES
    missing = assigned - all_known
    if missing:
        raise ValueError(f"These lakes were assigned but not found in your data: {sorted(missing)}")

    train_pool = [t for lk in TRAIN_LAKES for t in tiles_by_lake[lk]]
    val_pool   = [t for lk in VAL_LAKES   for t in tiles_by_lake[lk]]
    test_pool  = [t for lk in TEST_LAKES  for t in tiles_by_lake[lk]]

    train = build_split_balanced(train_pool, TARGET_TRAIN)
    val   = build_split_balanced(val_pool, TARGET_VAL)
    test  = build_split_balanced(test_pool, TARGET_TEST)

    return train, val, test

# ---------------------------
# 6) RUN
# ---------------------------
make_dirs(OUT_ROOT)

tiles_by_lake = collect_tiles_grouped_by_lake()
print("Lakes found:", sorted(tiles_by_lake.keys()))
print("Tiles per lake:", {k: len(v) for k, v in tiles_by_lake.items()})

train_tiles, val_tiles, test_tiles = build_splits_from_lakes(tiles_by_lake)

print("Final Train/Val/Test:", len(train_tiles), len(val_tiles), len(test_tiles))
summarise_split("TRAIN", train_tiles)
summarise_split("VAL  ", val_tiles)
summarise_split("TEST ", test_tiles)

save_tiles_with_rgb(train_tiles, "training")
save_tiles_with_rgb(val_tiles, "validation")
save_tiles_with_rgb(test_tiles, "test")

print("✅ Done. Saved to:", OUT_ROOT)
print("RGB previews saved under: <split>/rgb_png/")


Reading + aligning: darbandikhan_2021
Reading + aligning: dukan_2021
Reading + aligning: habbaniyah_2021
Reading + aligning: razzaza_2021
Reading + aligning: tharthar_2021
Lakes found: ['darbandikhan', 'dukan', 'habbaniyah', 'razzaza', 'tharthar']
Tiles per lake: {'darbandikhan': 35, 'dukan': 56, 'habbaniyah': 63, 'razzaza': 80, 'tharthar': 396}
Final Train/Val/Test: 250 56 20
TRAIN: n=250 | lakes=['habbaniyah', 'razzaza', 'tharthar'] | years=[2021] | non-empty=65.6% | heavy(>=0.15)=34.0% | wf_mean=0.288
VAL  : n=56 | lakes=['dukan'] | years=[2021] | non-empty=53.6% | heavy(>=0.15)=21.4% | wf_mean=0.145
TEST : n=20 | lakes=['darbandikhan'] | years=[2021] | non-empty=60.0% | heavy(>=0.15)=20.0% | wf_mean=0.071
✅ Done. Saved to: /content/drive/MyDrive/Iraq_Lake/Iraq_Lake_s2_water_seg
RGB previews saved under: <split>/rgb_png/


In [7]:
!ls -lah "/content/drive/MyDrive/Iraq_Lake/Iraq_Marshes_s2_water_seg"
!ls -lah "/content/drive/MyDrive/Iraq_Lake/Iraq_Marshes_s2_water_seg/validation/images" | head
!ls -lah "/content/drive/MyDrive/Iraq_Lake/Iraq_Marshes_s2_water_seg/test/images" | head


total 12K
drwx------ 4 root root 4.0K Feb  3 13:06 test
drwx------ 4 root root 4.0K Feb  3 13:06 training
drwx------ 4 root root 4.0K Feb  3 13:06 validation
total 225M
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c000.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c001.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c002.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c003.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c004.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c005.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c006.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r000_c007.npy
-rw------- 1 root root 4.1M Feb  3 13:08 dukan_2021_r001_c000.npy
total 81M
-rw------- 1 root root 4.1M Feb  3 13:08 darbandikhan_2021_r000_c003.npy
-rw------- 1 root root 4.1M Feb  3 13:08 darbandikhan_2021_r000_c004.npy
-rw------- 1 root root 4.1M Feb  3 13:08 darbandikhan_2021_r000_c005.npy
-rw-----